In [1]:
import math
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================
symbols = [
    "CDE", "BSX", "CELH", "COO", "INSM",
    "DRS", "ABT", "CLX", "HL", "OXY",
    "APA", "FSLR", "CF", "BF-B", "SPXC"
]

portfolio_value = 20_000
weight_per_symbol = 0.05
capital_per_symbol = portfolio_value * weight_per_symbol   # $1,000 per symbol

days_to_expiry_target = 75
days_held = 3   # simulate buying 2 trading days ago and exiting today
risk_free_rate = 0.04
otm_pct = 0.05

use_chain_iv = True
use_realized_vol_fallback = True

# final fallback if neither chain IV nor realized vol is usable
iv_overrides = {
    "CDE": 0.55,
    "BSX": 0.30,
    "CELH": 0.60,
    "COO": 0.28,
    "INSM": 0.50,
    "DRS": 0.32,
    "ABT": 0.25,
    "CLX": 0.24,
    "HL": 0.55,
    "OXY": 0.38,
    "APA": 0.42,
    "FSLR": 0.50,
    "CF": 0.35,
    "BF-B": 0.25,
    "SPXC": 0.30,
}
implied_vol_default = 0.35

# =========================================================
# BLACK-SCHOLES
# =========================================================
def bs_call_price(S, K, T, r, sigma):
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        return max(S - K, 0.0)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)

def choose_target_strike(spot, otm_pct=0.05):
    return spot * (1 + otm_pct)

def annualized_realized_vol(close_series, lookback=30):
    close_series = pd.Series(close_series).dropna()
    if len(close_series) < lookback + 1:
        return None
    rets = np.log(close_series / close_series.shift(1)).dropna().tail(lookback)
    if len(rets) < 5:
        return None
    vol = rets.std() * np.sqrt(252)
    if np.isnan(vol) or vol <= 0:
        return None
    return float(vol)

def nearest_expiration(ticker_obj, target_dte=75):
    try:
        expirations = ticker_obj.options
        if not expirations:
            return None, None

        today = pd.Timestamp.today().normalize()
        exp_dates = pd.to_datetime(expirations)
        dtes = [(exp.strftime("%Y-%m-%d"), int((exp.normalize() - today).days)) for exp in exp_dates]

        eligible = [(exp, dte) for exp, dte in dtes if dte >= 60]
        pool = eligible if eligible else dtes
        chosen = min(pool, key=lambda x: abs(x[1] - target_dte))
        return chosen
    except Exception:
        return None, None

def get_chain_call_info(symbol, spot, target_dte=75, otm_pct=0.05):
    """
    Returns:
      expiration_str, actual_dte, selected_strike, implied_vol, last_price, bid, ask
    """
    try:
        tkr = yf.Ticker(symbol)
        expiration_str, actual_dte = nearest_expiration(tkr, target_dte=target_dte)
        if expiration_str is None:
            return None, None, None, None, None, None, None

        chain = tkr.option_chain(expiration_str)
        calls = chain.calls.copy()
        if calls is None or calls.empty:
            return expiration_str, actual_dte, None, None, None, None, None

        target_strike = choose_target_strike(spot, otm_pct=otm_pct)

        calls = calls[calls["strike"] > 0].copy()
        if calls.empty:
            return expiration_str, actual_dte, None, None, None, None, None

        calls["distance"] = (calls["strike"] - target_strike).abs()
        row = calls.sort_values(["distance", "strike"]).iloc[0]

        selected_strike = float(row["strike"])
        iv = row.get("impliedVolatility", np.nan)
        last_price = row.get("lastPrice", np.nan)
        bid = row.get("bid", np.nan)
        ask = row.get("ask", np.nan)

        if pd.isna(iv) or iv <= 0:
            iv = None
        else:
            iv = float(iv)

        def clean(x):
            return None if pd.isna(x) else float(x)

        return expiration_str, actual_dte, selected_strike, iv, clean(last_price), clean(bid), clean(ask)
    except Exception:
        return None, None, None, None, None, None, None

def get_price_history(symbol, period="3mo"):
    try:
        df = yf.download(symbol, period=period, auto_adjust=True, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            close = df["Close"].squeeze()
        else:
            close = df["Close"].copy()
        return close.dropna()
    except Exception:
        return pd.Series(dtype=float)

def get_entry_exit_close(symbol, days_held=2, download_period="10d"):
    """
    Returns:
      entry_close, exit_close

    If days_held=2:
      entry_close = close from 2 trading days ago
      exit_close  = latest close
    """
    try:
        df = yf.download(symbol, period=download_period, auto_adjust=True, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            close = df["Close"].squeeze().dropna()
        else:
            close = df["Close"].dropna()

        if len(close) < days_held + 1:
            return None, None

        entry_close = float(close.iloc[-(days_held + 1)])
        exit_close = float(close.iloc[-1])
        return entry_close, exit_close
    except Exception:
        return None, None

def infer_iv(symbol, spot, target_dte=75, otm_pct=0.05):
    expiration_str, actual_dte, selected_strike, chain_iv, chain_last, chain_bid, chain_ask = (
        get_chain_call_info(symbol, spot, target_dte=target_dte, otm_pct=otm_pct)
        if use_chain_iv else (None, None, None, None, None, None, None)
    )

    iv_source = None
    sigma = None

    if chain_iv is not None and chain_iv > 0:
        sigma = chain_iv
        iv_source = "chain_iv"
    elif use_realized_vol_fallback:
        hist_close = get_price_history(symbol, period="3mo")
        rv = annualized_realized_vol(hist_close, lookback=30)
        if rv is not None and rv > 0:
            sigma = max(rv * 1.10, 0.15)
            iv_source = "realized_vol_x1.10"

    if sigma is None:
        sigma = iv_overrides.get(symbol, implied_vol_default)
        iv_source = "override_default"

    return {
        "expiration": expiration_str,
        "actual_dte": actual_dte if actual_dte is not None else target_dte,
        "selected_strike": selected_strike,
        "sigma": float(sigma),
        "iv_source": iv_source,
        "chain_last": chain_last,
        "chain_bid": chain_bid,
        "chain_ask": chain_ask,
    }

# =========================================================
# MAIN SIMULATION
# =========================================================
rows = []

for symbol in symbols:
    entry_close, exit_close = get_entry_exit_close(symbol, days_held=days_held)

    if entry_close is None or exit_close is None:
        print(f"Skipping {symbol}: not enough close data")
        continue

    iv_info = infer_iv(
        symbol=symbol,
        spot=entry_close,
        target_dte=days_to_expiry_target,
        otm_pct=otm_pct,
    )

    strike = iv_info["selected_strike"]
    if strike is None or strike <= 0:
        strike = choose_target_strike(entry_close, otm_pct=otm_pct)
        if entry_close < 25:
            step = 0.5
        elif entry_close < 100:
            step = 2.5
        else:
            step = 5.0
        strike = math.ceil(strike / step) * step

    sigma = iv_info["sigma"]
    actual_dte = iv_info["actual_dte"] if iv_info["actual_dte"] is not None else days_to_expiry_target

    T_entry = max(actual_dte, 1) / 365.0
    T_exit = max(actual_dte - days_held, 1) / 365.0

    # Use Black-Scholes for BOTH entry and exit so the pricing basis is consistent
    option_entry = bs_call_price(
        S=entry_close,
        K=strike,
        T=T_entry,
        r=risk_free_rate,
        sigma=sigma
    )

    option_exit = bs_call_price(
        S=exit_close,
        K=strike,
        T=T_exit,
        r=risk_free_rate,
        sigma=sigma
    )

    pricing_source = "bs_both"

    contract_entry_cost = option_entry * 100
    contract_exit_value = option_exit * 100

    if contract_entry_cost <= 0:
        print(f"Skipping {symbol}: zero option entry cost")
        continue

    contracts = int(capital_per_symbol // contract_entry_cost)
    if contracts < 1:
        print(f"Skipping {symbol}: contract too expensive for capital target")
        continue

    capital_used = contracts * contract_entry_cost
    ending_value = contracts * contract_exit_value
    pnl = ending_value - capital_used

    option_return_pct = ((option_exit - option_entry) / option_entry * 100) if option_entry > 0 else np.nan
    stock_return_pct = (exit_close - entry_close) / entry_close * 100

    rows.append({
        "symbol": symbol,
        "entry_close": round(entry_close, 2),
        "exit_close": round(exit_close, 2),
        "days_held": int(days_held),
        "stock_return_%": round(stock_return_pct, 2),
        "expiration": iv_info["expiration"],
        "dte_entry": int(actual_dte),
        "dte_exit": int(max(actual_dte - days_held, 1)),
        "strike": round(float(strike), 2),
        "iv_used": round(float(sigma), 4),
        "iv_source": iv_info["iv_source"],
        "chain_bid": None if iv_info["chain_bid"] is None else round(iv_info["chain_bid"], 4),
        "chain_ask": None if iv_info["chain_ask"] is None else round(iv_info["chain_ask"], 4),
        "chain_last": None if iv_info["chain_last"] is None else round(iv_info["chain_last"], 4),
        "pricing_source": pricing_source,
        "option_entry": round(option_entry, 4),
        "option_exit": round(option_exit, 4),
        "option_return_%": round(option_return_pct, 2),
        "contracts": int(contracts),
        "capital_target": round(capital_per_symbol, 2),
        "capital_used": round(capital_used, 2),
        "ending_value": round(ending_value, 2),
        "pnl": round(pnl, 2),
    })

df = pd.DataFrame(rows)

if df.empty:
    print("No trades simulated.")
else:
    df = df.sort_values("option_return_%", ascending=False)

    total_capital_used = df["capital_used"].sum()
    total_ending_value = df["ending_value"].sum()
    total_pnl = df["pnl"].sum()
    portfolio_return_pct = (total_pnl / total_capital_used * 100) if total_capital_used > 0 else np.nan

    print(df.to_string(index=False))
    print("\n" + "=" * 100)
    print(f"Days held:          {days_held}")
    print(f"Total capital used: ${total_capital_used:,.2f}")
    print(f"Total ending value: ${total_ending_value:,.2f}")
    print(f"Total PnL:          ${total_pnl:,.2f}")
    print(f"Portfolio return:   {portfolio_return_pct:.2f}%")

Skipping INSM: contract too expensive for capital target
Skipping FSLR: contract too expensive for capital target
Skipping CF: contract too expensive for capital target
Skipping SPXC: contract too expensive for capital target
symbol  entry_close  exit_close  days_held  stock_return_% expiration  dte_entry  dte_exit  strike  iv_used iv_source  chain_bid  chain_ask  chain_last pricing_source  option_entry  option_exit  option_return_%  contracts  capital_target  capital_used  ending_value     pnl
   BSX        59.52       62.61          3            5.19 2026-07-17         84        81    60.0   0.4146  chain_iv       5.60       6.30        6.08        bs_both        4.7467       6.4815            36.55          2          1000.0        949.34       1296.29  346.95
  CELH        32.69       34.92          3            6.81 2026-07-17         84        81    35.0   0.6968  chain_iv       4.55       4.70        4.67        bs_both        3.5454       4.6500            31.16          2     